# DS2: is the late "intervention" signal real?
The DS runs found that copying the fact word ("icy") from the **intervention** story acts like the ordinary patch (seeing) through most of the network, and shifts part of the way towards doing only in the last few layers where patches work. That was found, not predicted. This notebook tests it properly, with predictions written first (`prediction_before_DS2.txt`):

1. **Fresh items:** variants 0–15 of the same valid scenarios, never used for patching before.
2. **Single-layer and column patches.** Column = the fact word is replaced at *every* layer from 0 to L, so later tokens read the source version of the word all the way up.
3. **Reverse patch:** the *see* version of the word is written into the *do* story. If "how the fact came about" is held at the cause words, the do story should stay do-like (D near 1).

Scores as before: filler-corrected shifts against the neutral base story, r = Δcause/Δeffect, D = (r_see − r_edit)/(r_see − r_do), paired differences with bootstrap CIs and a sign-flip permutation test.

In [ ]:
# Kaggle setup: write the story files (embedded below) and work in /kaggle/working/see-vs-do
import os, gzip, base64
os.makedirs("/kaggle/working/see-vs-do", exist_ok=True)
os.chdir("/kaggle/working/see-vs-do")
DATA = {'pairs_chain.csv': '...', 'stories_chain.csv': '...'}
for name, b in DATA.items():
    open(name, "wb").write(gzip.decompress(base64.b64decode(b)))
OUT = "results"
os.makedirs(OUT, exist_ok=True)
import torch
print("working in", os.getcwd(), "| files:", sorted(os.listdir(".")), "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---------- Setup: model, tokenizer, settings
import gc, os, time, torch, numpy as np, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_grad_enabled(False)

NAME = "Qwen/Qwen3-4B-Base"
DEVICE = "cuda"
BS = 16                         # batch size; lower to 8 if you get out-of-memory errors
TRAIN_VARIANTS = range(0, 16)   # used only to learn directions
EVAL_VARIANTS = range(16, 20)   # held out; used for every score. Raise to range(16, 24) for the final run
LAYER_STEP = 4                  # plan of record: every 4th layer
ALPHAS = [1.0, 2.0, 4.0]        # edit strength, in units of the (true - false) difference vector
POSITIONS = ["mediator", "after", "answer"]

for v in ["lm", "model", "W32"]:
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load(dtype):
    m = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map=DEVICE).eval()
    return m, m.get_output_embeddings().weight.float()     # float32 output layer

lm, W32 = load(torch.float16)
N_LAYERS = lm.config.num_hidden_layers
LAYERS = list(range(0, N_LAYERS, LAYER_STEP))
print("layers to sweep:", LAYERS)

YES = tok(" Yes", add_special_tokens=False).input_ids
NO = tok(" No", add_special_tokens=False).input_ids
assert len(YES) == 1 and len(NO) == 1
YES, NO = YES[0], NO[0]

In [ ]:
# ---------- Prompt building, token positions, belief with optional edit
import ast
FEWSHOT = (
    "Story: The kettle was on the stove and the water started to bubble.\n"
    "Question: Was the water hot?\nAnswer: Yes\n\n"
    "Story: Maya left her umbrella at home and the sky stayed clear all day.\n"
    "Question: Did Maya get wet from rain?\nAnswer: No\n\n"
)
STORY_OFFSET = len(FEWSHOT) + len("Story: ")

def prompt(story, q):
    return f"{FEWSHOT}Story: {story}\nQuestion: {q}\nAnswer:"

def encode(stories, qs, spans):
    """Tokenise a batch (left-padded) and find, per row, the last token of the key sentence."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True,
              return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")
    am = enc["attention_mask"]
    key_idx = []
    for i, (a, b) in enumerate(spans):
        a, b = a + STORY_OFFSET, b + STORY_OFFSET
        hit = ((offs[i, :, 0] < b) & (offs[i, :, 1] > a) & (am[i] == 1)).nonzero().flatten()
        assert len(hit) > 0, "key sentence not found in tokens"
        key_idx.append(hit[-1].item())
    return enc, torch.tensor(key_idx)

def position_mask(key_idx, T, mode):
    B = len(key_idx)
    m = torch.zeros(B, T)
    for i, k in enumerate(key_idx.tolist()):
        if mode == "mediator":
            m[i, k] = 1
        elif mode == "after":
            m[i, k + 1:T - 1] = 1     # tokens after the key sentence, excluding the final token
        elif mode == "answer":
            m[i, T - 1] = 1
    return m

def _layer_out(out):
    return out[0] if isinstance(out, tuple) else out

def belief(stories, qs, spans, edit=None, bs=BS):
    """log P(Yes) - log P(No). edit = dict(layer=int, vecs=[N, H] tensor, mode=str) or None."""
    res = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handle = None
        if edit is not None:
            T = enc["input_ids"].shape[1]
            mask = position_mask(key_idx, T, edit["mode"]).to(DEVICE, lm.dtype)
            vecs = edit["vecs"][i:i+bs].to(DEVICE, lm.dtype)
            def hook(mod, inp, out, mask=mask, vecs=vecs):
                h = _layer_out(out) + mask[:, :, None] * vecs[:, None, :]
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handle = lm.model.layers[edit["layer"]].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            if handle is not None:
                handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

def key_states(stories, qs, spans, layers, bs=BS):
    """Residual stream (output of each layer in `layers`) at the last token of the key sentence."""
    out = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        rows = torch.arange(len(key_idx))
        out.append(torch.stack([hs[L + 1][rows, key_idx.to(DEVICE)].float().cpu() for L in layers], 1))
    return torch.cat(out)        # [N, len(layers), H]

# float16 overflow check; fall back to bfloat16 if needed
t = belief(["Priya ate a huge lunch and felt full."], ["Was Priya full?"], [[0, 5]])
if not np.isfinite(t).all():
    print("float16 overflow -> bfloat16")
    del lm, W32; gc.collect(); torch.cuda.empty_cache()
    lm, W32 = load(torch.bfloat16)
print("dtype:", lm.dtype)

d = pd.read_csv("stories_chain.csv")
d["variant"] = d.item_id.str.rsplit("_", n=1).str[1].astype(int)
for c in [c for c in d.columns if c.startswith("span_")]:
    d[c] = d[c].apply(ast.literal_eval)
train = d[d.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
ev = d[d.variant.isin(EVAL_VARIANTS)].reset_index(drop=True)
SCEN = sorted(d.scenario_id.unique())
print(f"{len(SCEN)} scenarios | train items {len(train)} | eval items {len(ev)}")

In [ ]:
# ---------- Settings
import glob, matplotlib.pyplot as plt
TAG = "DS2_qwen"
SOFTCAP = lambda x: x          # Qwen and Llama do not cap their logits
VALID = ['cf_blank_reprint', 'cf_breadbag_throw', 'cf_cast_ask', 'cf_cat_vet', 'cf_class_free', 'cf_dark_lights', 'cf_drawers_tidy', 'cf_grin_notice', 'cf_rash_itch', 'cf_roof_animals', 'cf_shelves_leave', 'cf_silent_dinner', 'cf_trains_home', 'cf_vegetables_cook', 'cf_waves_beach', 'ch_ash_flights', 'ch_beam_floor', 'ch_cracks_leave', 'ch_crops_income', 'ch_dents_repair', 'ch_hand_write', 'ch_ice_bus', 'ch_lake_fish', 'ch_river_fields', 'ch_sea_swim', 'ch_smell_leave', 'ch_stain_repair', 'ch_tree_route', 'ch_voice_sing', 'ch_vomit_rest']   # fixed from the earlier qwen runs
PATCH_LAYERS = [0, 6, 9, 12, 13, 14, 15, 16, 17]
LAYERS = list(range(0, max(PATCH_LAYERS) + 1))      # states are recorded at every layer up to the last patched one
FILLERS = ["Was there a cat nearby?", "Did someone in the story have a brother?", "Was a radio playing?"]
PROBES = {"cause": "q_cause", "effect": "q_eff", "med": "q_med", **{f"f{i}": f"q_f{i}" for i in range(len(FILLERS))}}

P = pd.read_csv("pairs_chain.csv")
for c in [c for c in P.columns if c.startswith(("span_", "wspan_"))]:
    P[c] = P[c].apply(ast.literal_eval)
P = P[P.scenario_id.isin(VALID) & P.variant.between(0, 15)].reset_index(drop=True)
for i, f in enumerate(FILLERS):
    P[f"q_f{i}"] = f
def do_word_span(r):
    wa, wb = r.wspan_see
    w = r.see[wa:wb]
    k = r.do.rfind(w)
    assert k >= r.span_do[0] and r.do[k + len(w):].startswith("."), (r.item_id, w)
    return [k, k + len(w)]
P["wspan_do"] = P.apply(do_word_span, axis=1)
P["span_do_s"] = P.apply(lambda r: [r.wspan_do[0], r.span_do[1]], axis=1)
print(f"{P.scenario_id.nunique()} scenarios, {len(P)} items | patch layers {PATCH_LAYERS}")

## Step 1: reference (seeing and doing from text) on these items

In [ ]:
path = f"{OUT}/{TAG}_reference_items.csv"
if os.path.exists(path):
    ref = pd.read_csv(path)
else:
    ref = P[["scenario_id", "item_id"]].copy()
    for cond in ["base", "see", "do", "false"]:
        for p, q in PROBES.items():
            ref[f"{p}_{cond}"] = belief(P[cond].tolist(), P[q].tolist(), P[f"span_{cond}"].tolist())
    ref.to_csv(path, index=False)
nf = len(FILLERS)
for cond in ["see", "do", "false"]:
    fill = sum(ref[f"f{i}_{cond}"] - ref[f"f{i}_base"] for i in range(nf)) / nf
    ref[f"dfill_{cond}"] = fill
    for p in ["cause", "effect", "med"]:
        ref[f"d{p}C_{cond}"] = ref[f"{p}_{cond}"] - ref[f"{p}_base"] - fill
sc = ref.groupby("scenario_id")[[c for c in ref.columns if c.startswith("d")]].mean()
sc["r_see"] = sc.dcauseC_see / sc.deffectC_see
sc["r_do"] = sc.dcauseC_do / sc.deffectC_do
sc["valid_now"] = ((sc.dcauseC_see > 0.5) & (sc.deffectC_see > 0.5) & (sc.deffectC_do > 0.5) & (sc.r_see - sc.r_do > 0.25))
sc.to_csv(f"{OUT}/{TAG}_reference_scenarios.csv")
print(f"{int(sc.valid_now.sum())} / {len(sc)} scenarios also pass the filters on these fresh variants (all are used, as fixed in advance)")

## Step 2: record sources, then forward (into the base story) and reverse (into the do story) patches

In [ ]:
def key_positions(stories, qs, spans, wspans):
    """Per row: index of the last token of the key word, and of the sentence end."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True, return_offsets_mapping=True)
    offs = enc.pop("offset_mapping"); am = enc["attention_mask"]
    wl, se = [], []
    for i, ((a, b), (wa, wb)) in enumerate(zip(spans, wspans)):
        def last_hit(x, y):
            x, y = x + STORY_OFFSET, y + STORY_OFFSET
            h = ((offs[i, :, 0] < y) & (offs[i, :, 1] > x) & (am[i] == 1)).nonzero().flatten()
            assert len(h) > 0
            return h[-1].item()
        wl.append(last_hit(wa, wb)); se.append(last_hit(a, b))
    return enc, wl, se

def record(stories, qs, spans, wspans, bs=BS):
    """States at [key-word-last ... sentence-end] for every layer in LAYERS. Returns list of [L, n_pos, H] tensors."""
    out = []
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        for r in range(len(wl)):
            out.append(torch.stack([hs[L + 1][r, wl[r]:se[r] + 1].float().cpu() for L in LAYERS]))
    return out

def belief_patch(stories, qs, spans, wspans, src, li, mode, bs=BS):
    """Run the target stories with states replaced at layer LAYERS[li]. src[row] = [L, n_pos, H]."""
    res, skipped = [], 0
    L = LAYERS[li]
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        items = []
        for r in range(len(wl)):
            s_states = src[i + r][li]                         # [n_pos_src, H]
            n_tgt = se[r] - wl[r] + 1
            if mode == "word":
                items.append(([wl[r]], s_states[:1]))
            elif s_states.shape[0] == n_tgt:
                items.append((list(range(wl[r], se[r] + 1)), s_states))
            else:
                items.append(None); skipped += 1
        def hook(mod, inp, out, items=items):
            h = _layer_out(out).clone()
            for r, it in enumerate(items):
                if it is not None:
                    h[r, it[0]] = it[1].to(h.device, h.dtype)
            return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
        handle = lm.model.layers[L].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        b = (logp[:, YES] - logp[:, NO]).tolist()
        res += [np.nan if it is None else v for it, v in zip(items, b)]
    return np.array(res), skipped


def belief_patch_column(stories, qs, spans, wspans, src, upto, bs=BS):
    """Word patch at EVERY layer 0..LAYERS[upto]: the word's state is replaced by the source's at each of those layers."""
    res = []
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handles = []
        for li in range(upto + 1):
            vec = torch.stack([src[i + r][li][0] for r in range(len(wl))])
            def hook(mod, inp, out, vec=vec, wl=wl):
                h = _layer_out(out).clone()
                h[torch.arange(len(wl)), torch.tensor(wl)] = vec.to(h.device, h.dtype)
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handles.append(lm.model.layers[LAYERS[li]].register_forward_hook(hook))
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            for hd in handles:
                hd.remove()
        logp = torch.log_softmax(SOFTCAP(h @ W32.T), dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

t0 = time.time()
q0 = P.q_med.tolist()
SRC = {"true": record(P.see.tolist(), q0, P.span_see.tolist(), P.wspan_see.tolist()),
       "do": record(P.do.tolist(), q0, P.span_do_s.tolist(), P.wspan_do.tolist()),
       "false": record(P.false.tolist(), q0, P.span_false.tolist(), P.wspan_false.tolist())}
other = {s: VALID[(VALID.index(s) + 1) % len(VALID)] for s in VALID}
lookup = {(r.scenario_id, r.variant): i for i, r in P.iterrows()}
U = P.loc[[lookup[(other[r.scenario_id], r.variant)] for _, r in P.iterrows()]]
SRC["unrelated"] = record(U.see.tolist(), U.q_med.tolist(), U.span_see.tolist(), U.wspan_see.tolist())
print(f"recorded sources for {len(P)} items at {len(LAYERS)} layers ({(time.time() - t0) / 60:.1f} min)")

B = {p: belief(P.base.tolist(), P[q].tolist(), P.span_base.tolist()) for p, q in PROBES.items()}
JOBS = [("forward", "single", k) for k in ["true", "do", "false", "unrelated"]] + [("forward", "column", k) for k in ["true", "do"]] \
     + [("reverse", "single", "true"), ("reverse", "column", "true")]
for L in PATCH_LAYERS:
    li = LAYERS.index(L)
    path = f"{OUT}/{TAG}_sweep_layer{L:02d}.csv"
    if os.path.exists(path):
        print("skip layer", L); continue
    rows = []
    for direction, mode, kind in JOBS:
        if direction == "forward":
            st, sp, wsp = P.base.tolist(), P.span_base.tolist(), P.wspan_base.tolist()
        else:
            st, sp, wsp = P.do.tolist(), P.span_do_s.tolist(), P.wspan_do.tolist()
        r = P[["scenario_id", "item_id"]].copy()
        r["layer"], r["direction"], r["mode"], r["kind"] = L, direction, mode, kind
        for p, q in PROBES.items():
            if mode == "single":
                b, _ = belief_patch(st, P[q].tolist(), sp, wsp, SRC[kind], li, "word")
            else:
                b = belief_patch_column(st, P[q].tolist(), sp, wsp, SRC[kind], li)
            r[f"d{p}"] = b - B[p]                 # always measured against the neutral base story
        rows.append(r)
    pd.concat(rows).to_csv(path, index=False)
    print(f"layer {L} done ({(time.time() - t0) / 60:.1f} min)")

## Step 3: results

In [ ]:
rng = np.random.default_rng(0)
def boot_median(x, n=4000):
    x = pd.Series(x).dropna().values
    if len(x) < 3:
        return np.nan, np.nan
    m = np.median(rng.choice(x, (n, len(x))), axis=1)
    return np.percentile(m, 2.5), np.percentile(m, 97.5)
def signflip_p(x, n=20000):
    x = pd.Series(x).dropna().values
    if len(x) < 3:
        return np.nan
    obs = abs(np.mean(x))
    flips = rng.choice([-1, 1], (n, len(x)))
    return float((np.abs((flips * x).mean(1)) >= obs).mean())

sw = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/{TAG}_sweep_layer*.csv")))
sw["kind"] = sw["kind"].astype(str).str.lower()
sw["dfill"] = sw[[f"df{i}" for i in range(len(FILLERS))]].mean(axis=1)
for p in ["cause", "effect", "med"]:
    sw[f"d{p}C"] = sw[f"d{p}"] - sw.dfill
g = sw.groupby(["scenario_id", "layer", "direction", "mode", "kind"])[["dcauseC", "deffectC", "dmedC", "dfill"]].mean().reset_index()
g = g.merge(sc[["deffectC_see", "r_see", "r_do"]], left_on="scenario_id", right_index=True)
g["works"] = g.deffectC >= 0.5 * g.deffectC_see
g["D"] = np.where(g.works, (g.r_see - g.dcauseC / g.deffectC) / (g.r_see - g.r_do), np.nan)
g.to_csv(f"{OUT}/{TAG}_scenario_scores.csv", index=False)

# pass rule for the forward single-layer true patch
pr = []
for L, x in g[(g.direction == "forward") & (g["mode"] == "single")].groupby("layer"):
    t, u, f = (x[x.kind == k].set_index("scenario_id") for k in ["true", "unrelated", "false"])
    d = (t.dmedC - u.dmedC).dropna()
    pr.append(dict(layer=L, dmedC_true=t.dmedC.mean(), minus_unrel=d.mean(), share_wins=(d > 0).mean(), dfill=t.dfill.mean(),
                   dmedC_false=f.dmedC.mean(), passes=bool(t.dmedC.mean() >= 1 and d.mean() > .5 and (d > 0).mean() >= .7
                                                       and abs(t.dfill.mean()) < .5 and f.dmedC.mean() < 0)))
pr = pd.DataFrame(pr); pr.to_csv(f"{OUT}/{TAG}_pass.csv", index=False); display(pr.round(2))

rows = []
for (L, mode), x in g[g.direction == "forward"].groupby(["layer", "mode"]):
    w = x.pivot_table(index="scenario_id", columns="kind", values="D")
    if not {"true", "do"} <= set(w.columns):
        continue
    d = (w["do"] - w["true"]).dropna(); lo, hi = boot_median(d)
    tl, th = boot_median(w["true"]); dl, dh = boot_median(w["do"])
    rows.append(dict(test="forward", layer=L, mode=mode, D_see_src=w["true"].median(), see_lo=tl, see_hi=th, D_do_src=w["do"].median(),
                     do_lo=dl, do_hi=dh, diff=d.median(), diff_lo=lo, diff_hi=hi, p_signflip=signflip_p(d), n_paired=len(d)))
for (L, mode), x in g[g.direction == "reverse"].groupby(["layer", "mode"]):
    lo, hi = boot_median(x.D)
    rows.append(dict(test="reverse", layer=L, mode=mode, D_rev=x.D.median(), rev_lo=lo, rev_hi=hi, n=int(x.D.notna().sum()),
                     works=x.works.mean()))
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/{TAG}_summary.csv", index=False)
display(res.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, mode in zip(axes[:2], ["single", "column"]):
    x = res[(res.test == "forward") & (res["mode"] == mode)]
    ax.errorbar(x.layer - .1, x.D_see_src, yerr=[x.D_see_src - x.see_lo, x.see_hi - x.D_see_src], fmt="o-", capsize=3, label="source: see story")
    ax.errorbar(x.layer + .1, x.D_do_src, yerr=[x.D_do_src - x.do_lo, x.do_hi - x.D_do_src], fmt="o-", capsize=3, color="C3", label="source: do story")
    ax.axhline(0, c="k", lw=.5); ax.axhline(1, c="k", lw=.5); ax.set_title(f"forward, {mode} patch"); ax.set_xlabel("layer"); ax.legend()
x = res[res.test == "reverse"]
for mode, c in [("single", "C0"), ("column", "C1")]:
    y = x[x["mode"] == mode]
    axes[2].errorbar(y.layer, y.D_rev, yerr=[y.D_rev - y.rev_lo, y.rev_hi - y.D_rev], fmt="o-", capsize=3, color=c, label=f"{mode} patch")
axes[2].axhline(0, c="k", lw=.5); axes[2].axhline(1, c="k", lw=.5); axes[2].set_title("reverse: see-word into the do story"); axes[2].legend()
plt.savefig(f"{OUT}/{TAG}_results.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results_DS2_qwen", "zip", OUT)
print("saved /kaggle/working/results_DS2_qwen.zip")